https://verl.readthedocs.io/en/latest/start/install.html#installation-from-docker
- docker 相对基础的概念、命令
    - 镜像与容器
    - 磁盘挂载
    - 宿主机与容器
- 基于 docker 的 verl 安装、开发等
    - cursor/vscode 连接 verl 容器，像是在本地开发一样
    - 持久化开发

```
docker run -it --rm \
-v "$PWD":/work -w /work \
node:20 bash
```
- cli 参数
    - `-it`: 交互式终端
    - `--rm`：容器生命周期的“阅后即焚”
        - `docker ps -a --filter "ancestor=node:20"`
    - `-v "$PWD":/work`：卷挂载 (Bind Mount)。将当前宿主机的目录（$PWD）映射到容器内的 /work 目录。
    - `-w /work`: 设定工作目录
    - `node:20`: 指定镜像版本
    - `bash`: 容器启动后运行的第一个进程。这里进入了 Bash Shell，等待用户输入指令。
- 这种模式在工程实践中被称为 "Ephemeral Environment"（临时环境）。它的核心价值在于：
    - 环境隔离：如果开发者的电脑上安装的是 Node 14，但项目需要 Node 20，使用此命令可以直接获得所需环境，无需使用 nvm 等工具反复切换。
    - 干净整洁：通过 --rm，所有的环境依赖、缓存或临时文件在退出后都会消失，只留下挂载目录里的代码和生成物（如 dist 或 node_modules）。
- `docker run --rm -v "$PWD":/work -w /work node:20`
    - 容器会瞬间启动，然后立即退出并被自动销毁，整个过程不会产生任何实质性的运行效果。

### concepts 101

- 镜像（Image）： 是静态的、只读的模板。如果用面向对象编程来类比，镜像是定义好的“类（Class）”；如果用建筑工程类比，镜像是“设计图纸”。构建镜像的常用命令是 docker build 或 docker commit。
- 容器（Container）： 是镜像的运行实体。它带有独立的状态，且顶层是可读写的。对应面向对象中的“实例化对象（Object）”，或是根据图纸真实建好的“房屋”。
    - docker start/stop 的对象是容器（显然非静态的镜像）
- `docker run = docker create + docker start`
    - docker create 基于镜像创建容器；
    - docker start 启动这个容器；
- 磁盘挂载
    - 绑定挂载（bind mounts）：`-v /home/user/downloads:/app/data`
        - 将宿主机的某个确切目录（例如 /home/user/downloads）强行映射到容器内部某个目录（例如 /app/data）
        - 如果配置了绑定挂载，当程序在容器的 /app/data 中下载文件时，该文件会立刻且直接出现在宿主机的 /home/user/downloads 目录下。二者指向的是同一块物理磁盘空间。

### 保持修改

- 启动时，去掉 `--rm`
    - `docker run -it --name codexbox -v "$PWD":/work -w /work node:20 bash`
- `docker start -ai codexbox`

### 常用命令

- docker start -ai
    - 唤醒并接管
    - `-ai`: a, attach; i, interactive;
    - 不加 `-ai` 会进入后台 detached mode
```bash
docker start verl
docker exec -it verl bash
```
- docker exec：在一个“已经在运行中”的容器内部，额外启动一个新的命令或进程。
    - `docker exec -it <容器名> bash`

### 基于容器的开发（以 verl docker 为例）

- https://verl.readthedocs.io/en/latest/start/install.html#installation-from-docker
    - dockerfile
        - `docker build -f docker/Dockerfile.stable.vllm -t verl-vllm:local .`
- `docker create --runtime=nvidia --gpus all --net=host --shm-size="10g" --cap-add=SYS_ADMIN -v .:/workspace/verl --name verl <image:tag> sleep infinity`
    - 概念上 `<image:tag>` 才是镜像（image），`--name verl` 是容器（container）
        - `verl-ci-cn-beijing.cr.volces.com/verlai/verl:vllm012.dev3`
        - 以指定的镜像为基础，在镜像的只读层之上附加一个可写的“容器层”，并为其分配所需的各项资源（如文件系统、网络接口、数据卷挂载等），从而生成一个具有唯一 ID 的容器实例。需要重点注意的是，由该命令创建的容器处于 已创建（Created） 状态，但并未启动（未运行任何进程）。
    - `--net=host`: 使用 宿主机网络命名空间（Host network）。容器与宿主共享同一套网络栈：
    - `--shm-size="10g"`: 共享内存
        - `df -h /dev/shm` 容器内查看
    - `-v .:/workspace/verl`
        - 把宿主机当前目录 . 以 bind mount 的方式挂载到容器内的 /workspace/verl。
    - `--name verl`
        - 把容器命名为 verl，方便后续操作：
            - docker start verl
            - docker exec -it verl bash
            - docker logs verl
    - `sleep infinity`
        - sleep infinity 的核心作用是强行赋予容器一个永远不会结束的主进程，从而迫使容器无限期地保持“运行中（Up）”状态，充当一个常驻的后台开发环境。
            - docker start 之后才是永久的
            - 加与不加 sleep infinity 的根本区别在于：容器启动后是“长久驻留后台待命”，还是“执行完镜像默认任务后瞬间退出（Exited）”。
        - docker stop 和 docker start 的状态流转后，该容器依然会坚定不移地保持 sleep infinity 的性质，绝不会“遗忘”这一指令或退回镜像的初始状态。

```
启动：docker start verl
进入：docker exec -it verl bash
停止：docker stop verl
docker top verl # Display the running processes of a container
状态查看：
    docker ps -a | grep verl
资源查看：
    docker stats --no-stream verl
删除（谨慎）：
    docker rm -f verl
```
- 安装成功，且进入后
    - nvidia-smi
    - pip show torch
        - `python -c "import torch; print(torch.__version__)"`
    - pip show verl: 查看 verl 版本（需要首先在容器内 github 源码下载并安装）
        - `0.8.0.dev0`
    - pip show vllm

- vscode/cursor 连接
    - 注意插件要用 anysphere.remote-containers / anysphere.remote ssh（而不是 ms-vscode-remote.remote-containers）
        - ms-vscode-remote.remote-containers：微软 VS Code 官方扩展 ID。
        - anysphere.remote-containers：Cursor 的同类扩展（重映射/兼容版本）。在 Cursor 里通常应使用这个。
- 关于宿主机与容器的 gpu
    - 驱动共享，可分别执行 `nvidia-smi` 查看（例如版本 535.xx、550.xx 等内核模块）
    - 但容器内 CUDA 工具包 (Runtime API / libcudart.so / nvcc) —— 完全独立，由这个容器 Dockerfile 的 `FROM nvidia/cuda:12.9.1-devel-ubuntu22.04`
    - `--runtime=nvidia --gpus all`: 该参数指示 Docker 守护进程向该容器暴露宿主机上的 GPU 资源。all 表示分配所有可见的显卡。如果只需要特定的显卡，通常会写成 --gpus '"device=0,1"'。`--runtime=nvidia` 的含义是：让 Docker 用 nvidia-container-runtime（来自 NVIDIA Container Toolkit）来启动容器，而不是默认的 runc。这两个参数实现：容器使用的是宿主机内核里的 NVIDIA 驱动（/dev/nvidia* 设备和对应驱动库映射进容器）。

```
# 宿主机
ls -l /dev/nvidia*
docker info | grep -i runtime

# 容器
ls -l /dev/nvidia*
```

### misc

- 个性化创建 verl 容器

```
CACHE_REAL="$(readlink -f ~/.cache)"

docker create \
    --runtime=nvidia --gpus all --net=host --shm-size="10g" --cap-add=SYS_ADMIN \
    -v "$PWD:/workspace/verl" \
    --mount type=bind,src="$CACHE_REAL",dst=/root/.cache \
    -e HF_HOME=/root/.cache/huggingface \
    -e TRANSFORMERS_CACHE=/root/.cache/huggingface \
    --name verl-vllm \
    --hostname verl \
    verl-ci-cn-beijing.cr.volces.com/verlai/verl:vllm012.dev3 \
    sleep infinity
```


- verl training 相关

```
huggingface-cli login
huggingface-cli whoami

wandb login
```


- training
    - data prepare
    - training bash scripts